# Chapter XI — Advanced Topics

Topics beyond the core chapters: useful math, topological sort, Dijkstra's algorithm, hash table collision resolution, Rabin–Karp substring search, AVL trees, red–black trees, and MapReduce.

In [1]:
from __future__ import annotations

import heapq
import math
import random
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from itertools import combinations, groupby, permutations

## Useful math
- `1 + 2 + … + n = n(n+1)/2`
- `2⁰ + 2¹ + … + 2ⁿ = 2ⁿ⁺¹ - 1`
- A base-b number with k digits has up to `bᵏ` values, and `log_b(n)` digits.
- Permutations: `n!/(n-k)!`. Combinations: `C(n, k) = n!/(k!(n-k)!)`. Subsets: `2ⁿ`.

In [2]:
n = 50
assert sum(range(1, n + 1)) == n * (n + 1) // 2
assert sum(2**i for i in range(n + 1)) == 2 ** (n + 1) - 1
assert len(list(permutations(range(6), 3))) == math.perm(6, 3) == 120
assert len(list(combinations(range(6), 3))) == math.comb(6, 3) == 20
assert len(str(10**20)) == math.floor(math.log10(10**20)) + 1
print("math ok")

math ok


## Topological sort (DFS version)
Chapter 4 used Kahn's algorithm. The DFS version adds each node to the front of the output once all of its descendants are finished. A back edge (reaching a node still in progress) means there's a cycle.

In [3]:
def topo_sort_dfs(graph: dict[str, list[str]]) -> list[str]:
    UNVISITED, VISITING, DONE = 0, 1, 2
    state: dict[str, int] = defaultdict(int)
    order: deque[str] = deque()

    def visit(u: str) -> None:
        if state[u] == VISITING:
            raise ValueError("cycle")
        if state[u] == DONE:
            return
        state[u] = VISITING
        for v in graph.get(u, []):
            visit(v)
        state[u] = DONE
        order.appendleft(u)

    for u in list(graph):
        visit(u)
    return list(order)


g = {"shirt": ["tie", "belt"], "tie": ["jacket"], "pants": ["shoes", "belt"], "belt": ["jacket"], "socks": ["shoes"]}
order = topo_sort_dfs(g)
assert all(order.index(u) < order.index(v) for u, vs in g.items() for v in vs)
print(order)

['socks', 'pants', 'shoes', 'shirt', 'belt', 'tie', 'jacket']


## Dijkstra's algorithm
Shortest paths from one source in a weighted graph with no negative edges. Using a binary heap (and lazily skipping stale entries) gives `O((V + E) log V)`.

In [4]:
def dijkstra(graph: dict[str, dict[str, int]], src: str) -> tuple[dict[str, float], dict[str, str | None]]:
    dist: dict[str, float] = defaultdict(lambda: math.inf)
    prev: dict[str, str | None] = {src: None}
    dist[src] = 0
    pq = [(0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue  # stale
        for v, w in graph.get(u, {}).items():
            nd = d + w
            if nd < dist[v]:
                dist[v], prev[v] = nd, u
                heapq.heappush(pq, (nd, v))
    return dict(dist), prev


def path_to(prev: dict[str, str | None], target: str) -> list[str]:
    out: list[str] = []
    node: str | None = target
    while node is not None:
        out.append(node)
        node = prev[node]
    return out[::-1]


G = {"a": {"b": 5, "c": 3, "e": 2}, "b": {"d": 2}, "c": {"b": 1, "d": 1}, "d": {"a": 1, "g": 2, "h": 1},
     "e": {"a": 1, "h": 4, "i": 7}, "f": {"b": 3, "g": 1}, "g": {"c": 3, "i": 2}, "h": {"c": 2, "f": 2, "g": 2}, "i": {}}
dist, prev = dijkstra(G, "a")
assert dist["i"] == 8 and path_to(prev, "i") == ["a", "c", "d", "g", "i"]
print({k: dist[k] for k in sorted(dist)})

{'a': 0, 'b': 4, 'c': 3, 'd': 4, 'e': 2, 'f': 7, 'g': 6, 'h': 5, 'i': 8}


## Hash table collision resolution
- **Chaining** with linked lists (Chapters 1 and 7).
- **Chaining with BSTs**: `O(log n)` worst case (Java 8's HashMap turns long chains into trees).
- **Open addressing** with **linear probing**: on a collision, try the next slot. Deletions need tombstones.
- **Quadratic probing** and **double hashing** reduce clustering.

In [5]:
_TOMBSTONE = object()


class OpenAddressingTable:
    def __init__(self, capacity: int = 8, probing: str = "linear") -> None:
        self.slots: list = [None] * capacity
        self.size = 0
        self.probing = probing

    def _probe(self, key):
        n = len(self.slots)
        h1 = hash(key) % n
        h2 = 1 + (hash((key, "salt")) % (n - 1))  # double hashing step, never 0
        for i in range(n):
            match self.probing:
                case "linear": yield (h1 + i) % n
                case "quadratic": yield (h1 + i * i) % n
                case "double": yield (h1 + i * h2) % n

    def put(self, key, value) -> None:
        if (self.size + 1) / len(self.slots) > 0.5:
            self._resize()
        first_free = None
        for idx in self._probe(key):
            slot = self.slots[idx]
            if slot is None:
                target = idx if first_free is None else first_free
                self.slots[target] = (key, value)
                self.size += 1
                return
            if slot is _TOMBSTONE:
                if first_free is None:
                    first_free = idx
            elif slot[0] == key:
                self.slots[idx] = (key, value)
                return
        if first_free is not None:
            self.slots[first_free] = (key, value)
            self.size += 1
            return
        self._resize()
        self.put(key, value)

    def get(self, key):
        for idx in self._probe(key):
            slot = self.slots[idx]
            if slot is None:
                raise KeyError(key)
            if slot is not _TOMBSTONE and slot[0] == key:
                return slot[1]
        raise KeyError(key)

    def delete(self, key) -> None:
        for idx in self._probe(key):
            slot = self.slots[idx]
            if slot is None:
                raise KeyError(key)
            if slot is not _TOMBSTONE and slot[0] == key:
                self.slots[idx] = _TOMBSTONE
                self.size -= 1
                return
        raise KeyError(key)

    def _resize(self) -> None:
        old = [s for s in self.slots if s is not None and s is not _TOMBSTONE]
        self.slots = [None] * (len(self.slots) * 2 + 1)  # odd size helps double hashing
        self.size = 0
        for k, v in old:
            self.put(k, v)


for mode in ("linear", "quadratic", "double"):
    t = OpenAddressingTable(probing=mode)
    ref = {}
    for i in range(2000):
        k = random.randint(0, 500)
        if random.random() < 0.3 and k in ref:
            t.delete(k); del ref[k]
        else:
            t.put(k, i); ref[k] = i
    assert all(t.get(k) == v for k, v in ref.items()) and t.size == len(ref)
print("open addressing ok")

open addressing ok


## Rabin–Karp substring search
A rolling hash lets you check the next window in `O(1)` instead of `O(k)`, for `O(n + m)` expected time. It uses a polynomial hash modulo a large prime, and compares the actual text on a hash match to rule out collisions.

In [6]:
def rabin_karp(text: str, pattern: str, base: int = 256, mod: int = 1_000_000_007) -> list[int]:
    n, m = len(text), len(pattern)
    if m == 0 or m > n:
        return []
    high = pow(base, m - 1, mod)
    ph = th = 0
    for i in range(m):
        ph = (ph * base + ord(pattern[i])) % mod
        th = (th * base + ord(text[i])) % mod
    hits = []
    for i in range(n - m + 1):
        if ph == th and text[i:i + m] == pattern:
            hits.append(i)
        if i + m < n:
            th = ((th - ord(text[i]) * high) * base + ord(text[i + m])) % mod
    return hits


txt = "abracadabra abracadabra"
assert rabin_karp(txt, "abra") == [i for i in range(len(txt)) if txt.startswith("abra", i)]
print("rabin-karp ok", rabin_karp(txt, "cad"))

rabin-karp ok [4, 16]


## AVL tree
An AVL tree stores each node's height and keeps every balance factor in {-1, 0, 1} by rotating after inserts. That gives `O(log n)` height.

In [7]:
@dataclass(eq=False)
class AVLNode:
    key: int
    left: AVLNode | None = None
    right: AVLNode | None = None
    height: int = 1


def _h(n: AVLNode | None) -> int:
    return n.height if n else 0


def _update(n: AVLNode) -> None:
    n.height = 1 + max(_h(n.left), _h(n.right))


def _rotate_right(y: AVLNode) -> AVLNode:
    x = y.left
    y.left, x.right = x.right, y
    _update(y); _update(x)
    return x


def _rotate_left(x: AVLNode) -> AVLNode:
    y = x.right
    x.right, y.left = y.left, x
    _update(x); _update(y)
    return y


def avl_insert(n: AVLNode | None, key: int) -> AVLNode:
    if n is None:
        return AVLNode(key)
    if key < n.key:
        n.left = avl_insert(n.left, key)
    else:
        n.right = avl_insert(n.right, key)
    _update(n)
    balance = _h(n.left) - _h(n.right)
    if balance > 1:
        if key >= n.left.key:           # left-right case
            n.left = _rotate_left(n.left)
        return _rotate_right(n)
    if balance < -1:
        if key < n.right.key:           # right-left case
            n.right = _rotate_right(n.right)
        return _rotate_left(n)
    return n


def check_avl(n: AVLNode | None) -> int:
    if n is None:
        return 0
    lh, rh = check_avl(n.left), check_avl(n.right)
    assert abs(lh - rh) <= 1 and n.height == 1 + max(lh, rh)
    return n.height


def inorder(n):
    return inorder(n.left) + [n.key] + inorder(n.right) if n else []


root = None
for k in range(1, 1024):  # sorted inserts: worst case for a plain BST
    root = avl_insert(root, k)
assert check_avl(root) <= 1.44 * math.log2(1024) + 1 and inorder(root) == list(range(1, 1024))
print("AVL height for 1023 sorted inserts:", root.height)

AVL height for 1023 sorted inserts: 10


## Red–black tree
Red–black rules: every node is red or black, the root is black, a red node has no red child, and every root-to-leaf path passes through the same number of black nodes. Below is the **left-leaning** variant (Sedgewick's LLRB), which needs fewer cases than the textbook version and keeps the same guarantees.

In [8]:
RED, BLACK = True, False


@dataclass(eq=False)
class RBNode:
    key: int
    color: bool = RED
    left: RBNode | None = None
    right: RBNode | None = None


def _is_red(n: RBNode | None) -> bool:
    return n is not None and n.color is RED


def _rb_rotate_left(h: RBNode) -> RBNode:
    x = h.right
    h.right, x.left = x.left, h
    x.color, h.color = h.color, RED
    return x


def _rb_rotate_right(h: RBNode) -> RBNode:
    x = h.left
    h.left, x.right = x.right, h
    x.color, h.color = h.color, RED
    return x


def _flip(h: RBNode) -> None:
    h.color = not h.color
    h.left.color = not h.left.color
    h.right.color = not h.right.color


def rb_insert(root: RBNode | None, key: int) -> RBNode:
    def ins(h: RBNode | None) -> RBNode:
        if h is None:
            return RBNode(key)
        if key < h.key:
            h.left = ins(h.left)
        elif key > h.key:
            h.right = ins(h.right)
        if _is_red(h.right) and not _is_red(h.left):
            h = _rb_rotate_left(h)
        if _is_red(h.left) and _is_red(h.left.left):
            h = _rb_rotate_right(h)
        if _is_red(h.left) and _is_red(h.right):
            _flip(h)
        return h

    root = ins(root)
    root.color = BLACK
    return root


def black_height(n: RBNode | None) -> int:
    if n is None:
        return 1
    assert not (_is_red(n) and (_is_red(n.left) or _is_red(n.right))), "red node with red child"
    lh, rh = black_height(n.left), black_height(n.right)
    assert lh == rh, "unequal black heights"
    return lh + (n.color is BLACK)


def rb_height(n):
    return 0 if n is None else 1 + max(rb_height(n.left), rb_height(n.right))


rb = None
keys = random.sample(range(100_000), 5000)
for k in keys:
    rb = rb_insert(rb, k)
black_height(rb)
assert rb_height(rb) <= 2 * math.log2(5001)
print("red-black ok, height:", rb_height(rb), "for", len(keys), "keys")

red-black ok, height: 17 for 5000 keys


## MapReduce
The **map** step emits (key, value) pairs, the shuffle groups the pairs by key, and **reduce** combines each group. Below, a word count and an "average temperature per city" job run in parallel on a thread pool (a stand-in for a cluster of workers).

In [9]:
from concurrent.futures import ThreadPoolExecutor


def run_mapreduce(inputs, mapper, reducer, workers: int = 4):
    with ThreadPoolExecutor(workers) as ex:
        mapped = [kv for chunk in ex.map(lambda x: list(mapper(x)), inputs) for kv in chunk]
    mapped.sort(key=lambda kv: kv[0])
    groups = [(k, [v for _, v in g]) for k, g in groupby(mapped, key=lambda kv: kv[0])]
    with ThreadPoolExecutor(workers) as ex:
        return dict(ex.map(lambda kv: (kv[0], reducer(kv[0], kv[1])), groups))


docs = ["the cat sat", "the dog sat", "a cat and a dog"]
wc = run_mapreduce(docs, lambda d: ((w, 1) for w in d.split()), lambda k, vs: sum(vs))
assert wc == Counter(" ".join(docs).split())

readings = ["nyc,20", "sf,15", "nyc,24", "sf,17", "la,30"]
avg = run_mapreduce(readings, lambda r: [(r.split(",")[0], int(r.split(",")[1]))], lambda k, vs: sum(vs) / len(vs))
assert avg == {"la": 30.0, "nyc": 22.0, "sf": 16.0}
print(wc, avg)

{'a': 2, 'and': 1, 'cat': 2, 'dog': 2, 'sat': 2, 'the': 2} {'la': 30.0, 'nyc': 22.0, 'sf': 16.0}
